# 50 — Score the pre-op cohort and publish explanations

**Layer:** model → gold · **Engine:** PySpark for the read/write, Python for inference

This notebook closes the loop. It loads `@champion` from the registry, scores every
patient awaiting surgery, and writes **two** gold tables:

* `gold.patient_risk` — one calibrated probability per patient, plus the model version
  that produced it,
* `gold.risk_explanation` — the per-patient feature contributions, long-format.

Both are consumed by the Rayfin clinician app and by the Data Agent. Neither the app nor
the agent ever loads a model — they read governed tables, which is what makes the
permission story hold.

### Why precompute the explanations

You have two options for SHAP-at-serve, and the session plan flags it as a decision:

| | Precompute to gold (this notebook) | Compute at inference |
|---|---|---|
| Latency | ~0 ms — it is a table read | 200 ms – 2 s per patient |
| Demo risk on stage | low | a live SHAP call is a live SHAP call |
| Ad-hoc "what if" | needs a re-run | works immediately |
| Agent can query it | **yes** — explanations are rows | no |

Precompute wins for the cohort view, and it is the only option that lets the Data Agent
answer *"why is Margaret high risk?"* — the agent queries SQL, so the explanation has to
be a row, not a function call. Keep the live endpoint for the single-patient what-if in
segment 5; use these tables for everything else.

## 1. Load the champion model and its preprocessor

In [ ]:
# ---------------------------------------------------------------------------
# joblib pickles ProMsPreprocessor by reference (__main__.ProMsPreprocessor),
# not by value. Notebook 40 defined it in its own session; this session has
# never seen it, hence:
#     AttributeError: Can't get attribute 'ProMsPreprocessor' on <module '__main__'>
# Only the attributes come out of the pickle -- the methods come from here, so
# this definition MUST stay byte-identical to 40_train_register section 4.
# ---------------------------------------------------------------------------
import numpy as np, pandas as pd
import re
import miceforest as mf
from sklearn.preprocessing import OneHotEncoder

# Delta rejects ' ,;{}()\n\t=' in column names, and one-hot encoding turns the category
# "60 to 69" into `age_band_grouped_60 to 69` -- which fails the write in §5b with
# DELTA_INVALID_CHARACTERS_IN_COLUMN_NAMES.
#
# The rename belongs here, not at the Delta write. The estimator is fitted on whatever
# names this class emits, so sanitising at the source keeps the encoded matrix, the
# logged MLflow signature and `ml.train_matrix` in agreement. Rename only on the way
# into Delta and 42/43 read the table back under different names than the model was
# fitted with, and sklearn rejects the frame with a feature-name mismatch.
_DELTA_INVALID = re.compile(r"[ ,;{}()\n\t=]+")


def delta_safe(name):
    """`age_band_grouped_60 to 69` -> `age_band_grouped_60_to_69`."""
    return _DELTA_INVALID.sub("_", str(name)).strip("_")


class ProMsPreprocessor:
    """MICE imputation + one-hot encoding, fitted on train, applied everywhere."""

    def __init__(self, categorical_columns, random_state=77, mice_iterations=5):
        self.categorical_columns = list(categorical_columns)
        self.random_state = random_state
        self.mice_iterations = mice_iterations
        self.kernel_ = None
        self.encoder_ = None
        self.feature_names_ = None

    def __getstate__(self):
        """Make the fitted kernel picklable.

        miceforest keeps its mean-matching predictions as float16
        (_PRE_LINK_DATATYPE) and its own __getstate__ serialises them with
        df.to_parquet(). Parquet has no float16 type, so a default kernel raises
        ArrowNotImplementedError on pickle -- but that history is exactly what
        impute_new_data() needs at serving time, so we cannot simply drop it
        (save_all_iterations_data=False makes the kernel picklable and useless).

        Casting to float32 satisfies Parquet, costs a few MB, and does not change
        the imputation: the values feed a KD-tree nearest-neighbour lookup.
        """
        if self.kernel_ is not None:
            for store in (self.kernel_.candidate_preds, self.kernel_.imputation_values):
                for col, df in list(store.items()):
                    if (df.dtypes == "float16").any():
                        store[col] = df.astype("float32")
        return self.__dict__.copy()

    def _as_categorical(self, X):
        X = X.copy().replace([np.inf, -np.inf], np.nan)
        for c in self.categorical_columns:
            X[c] = X[c].astype("category")
        return X

    def fit(self, X, y=None):
        Xc = self._as_categorical(X).reset_index(drop=True)
        self.kernel_ = mf.ImputationKernel(data=Xc, random_state=self.random_state)
        self.kernel_.mice(
            iterations=self.mice_iterations,
            variable_parameters={
                # Rare categories need larger leaves or LightGBM refuses to split.
                # Only min_data_in_leaf: min_child_samples is a LightGBM alias for it, and
                # miceforest collapses aliases to the canonical name -- passing both means
                # one silently overwrites the other.
                "age_band_grouped":            {"min_data_in_leaf": 30},
                "living_arrangements_grouped": {"min_data_in_leaf": 10},
            },
        )
        imputed = self.kernel_.complete_data(dataset=0)

        self.encoder_ = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")
        self.encoder_.fit(imputed[self.categorical_columns])
        self._dtypes = imputed.dtypes.to_dict()
        self.feature_names_ = self._assemble(imputed).columns.tolist()
        return self

    def _assemble(self, imputed):
        numeric = imputed.drop(columns=self.categorical_columns)
        encoded = pd.DataFrame(
            self.encoder_.transform(imputed[self.categorical_columns]),
            columns=[delta_safe(c) for c in
                     self.encoder_.get_feature_names_out(self.categorical_columns)],
            index=imputed.index,
        )
        out = pd.concat([numeric, encoded], axis=1)
        # Two categories that differ only in a stripped character would collapse into one
        # column and silently merge two levels. Fail loudly instead.
        assert out.columns.is_unique, (
            f"delta_safe() collided columns: "
            f"{out.columns[out.columns.duplicated()].tolist()}"
        )
        return out

    def transform(self, X):
        Xc = self._as_categorical(X).reset_index(drop=True)
        Xc = Xc.astype({k: v for k, v in self._dtypes.items() if k in Xc.columns})
        imputed = self.kernel_.impute_new_data(Xc).complete_data(0)
        out = self._assemble(imputed)
        # Serving may hand us a single row missing a rare category — align to training shape.
        return out.reindex(columns=self.feature_names_, fill_value=0)

    def fit_transform(self, X, y=None):
        return self.fit(X, y).transform(X)

print("ProMsPreprocessor defined -- joblib.load can now resolve it.")


In [ ]:
import mlflow, mlflow.sklearn, joblib, json, os
import numpy as np, pandas as pd
from mlflow import MlflowClient
from pyspark.sql import functions as F

EXPERIMENT       = "knee-poor-outcome"
REGISTERED_MODEL = "knee-poor-outcome-ebm"
CHAMPION_LABEL   = "champion"   # a version tag, not an MLflow alias -- Fabric has no alias API
GOLD_COHORT      = "gold.knee_preop_cohort"
GOLD_RISK        = "gold.patient_risk"
GOLD_EXPLAIN     = "gold.risk_explanation"

# --- champion resolution ----------------------------------------------------
# Fabric's MLflow registry does not implement the alias endpoints
# (/api/2.0/mlflow/registered-models/alias -> 404), so `@champion` URIs and
# get_model_version_by_alias() are unavailable. Version tags work, so the
# champion pointer is the tag champion='true'. This helper tries the alias
# first so the code stays correct on a tenant where aliases do exist.
def resolve_champion(client, name):
    try:
        return client.get_model_version_by_alias(name, "champion")
    except Exception:
        pass
    versions = list(client.search_model_versions(f"name='{name}'"))
    if not versions:
        raise RuntimeError(f"No versions registered for {name} -- run 40_train_register first.")
    champions = [v for v in versions if v.tags.get("champion") == "true"]
    if champions:
        return max(champions, key=lambda v: int(v.version))
    latest = max(versions, key=lambda v: int(v.version))
    print(f"WARNING: no champion tag found; falling back to latest version {latest.version}")
    return latest


mlflow.set_experiment(EXPERIMENT)
client = MlflowClient()

mv = resolve_champion(client, REGISTERED_MODEL)
MODEL_VERSION = mv.version
print(f"{REGISTERED_MODEL} champion -> version {MODEL_VERSION} (run {mv.run_id})")

model = mlflow.sklearn.load_model(f"models:/{REGISTERED_MODEL}/{mv.version}")

# The preprocessor was logged alongside the model, so training and serving cannot drift.
local = client.download_artifacts(mv.run_id, "preprocessing")
pre = joblib.load(os.path.join(local, "preprocessor.joblib"))
with open(os.path.join(local, "feature_columns.json")) as f:
    schema = json.load(f)

assert pre.feature_names_ == schema["encoded_columns"], \
    "Loaded preprocessor disagrees with the logged feature schema -- 40 and 50 have drifted."

GOLD_FEATURE_COLUMNS = schema["gold_columns"]
print(f"Preprocessor loaded: {len(GOLD_FEATURE_COLUMNS)} gold columns -> "
      f"{len(schema['encoded_columns'])} encoded features")

## 2. Score the cohort

`predict_proba(...)[:, 1]` is the probability of the **positive class**, which here is the
*poor* outcome. Getting that polarity wrong is the single most dangerous off-by-one in the
whole pipeline, so the risk band is derived from it explicitly and asserted below.

## 2b. Incremental mode — scoring only new patients

Everything above scores the whole cohort, which is right for a rebuild and wrong for a
triggered pipeline. When this notebook runs on a schedule or on new-data arrival, re-scoring
150,000 unchanged patients to pick up the 40 that arrived overnight is waste, and it rewrites
`_scored_at_utc` on rows nobody touched — which destroys the audit answer to *"when was this
patient scored, and by which model version?"*

Set `INCREMENTAL = True` and the notebook scores only what is genuinely new:

* an episode with **no score yet**, or
* an episode whose score came from a **different model version** than the current
  `@champion` — so promoting a new model automatically re-scores the cohort.

The write becomes a `MERGE` rather than an overwrite, so untouched rows keep their original
timestamp and model version.

**This is the answer to "does triggered inference need its own notebook?"** — no. It needs
*this* notebook to be incremental and to have a trigger in front of it. A second notebook
would duplicate the preprocessing, and duplicated preprocessing is how training and serving
drift apart.

In [ ]:
INCREMENTAL = False      # set True for scheduled / triggered runs

if INCREMENTAL and spark.catalog.tableExists(GOLD_RISK):
    already = spark.sql(f"""
        SELECT episode_id FROM {GOLD_RISK}
        WHERE model_version = '{MODEL_VERSION}'
    """)
    to_score = (spark.table(GOLD_COHORT)
                     .join(already, on="episode_id", how="left_anti"))
    n_new = to_score.count()
    print(f"Incremental: {n_new:,} episodes need scoring "
          f"(new, or scored by a version other than v{MODEL_VERSION})")
    if n_new == 0:
        print("Nothing to do — every patient already has a current score.")
    cohort = to_score.toPandas()
else:
    cohort = spark.table(GOLD_COHORT).toPandas()
    print(f"Full rebuild: scoring all {len(cohort):,} episodes")

### Writing incrementally

`MERGE` instead of `overwrite`. Re-run this notebook after promoting a new `@champion` and it
back-fills every stale row without touching the fresh ones.

In [ ]:
from delta.tables import DeltaTable

def upsert(df, table, key="episode_id"):
    """MERGE on the key, preserving rows this run did not touch."""
    if not spark.catalog.tableExists(table):
        df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table)
        print(f"created {table}: {df.count():,} rows")
        return
    (DeltaTable.forName(spark, table).alias("t")
        .merge(df.alias("s"), f"t.{key} = s.{key}")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    print(f"merged into {table}: {df.count():,} rows affected")

# For explanations the key is (episode_id, rank) — six rows per patient.
def upsert_explanations(df, table):
    if not spark.catalog.tableExists(table):
        df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table)
        print(f"created {table}: {df.count():,} rows")
        return
    (DeltaTable.forName(spark, table).alias("t")
        .merge(df.alias("s"), "t.episode_id = s.episode_id AND t.rank = s.rank")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    print(f"merged into {table}: {df.count():,} rows affected")

# Use these in place of the overwrite writes in §5 when INCREMENTAL is True:
#     upsert(risk_sdf, GOLD_RISK)
#     upsert_explanations(explain_sdf, GOLD_EXPLAIN)
print("Upsert helpers ready.")

## 2c. Wiring the trigger

The notebook is the *task*; something else has to *fire* it. Three options in Fabric, in
increasing order of how much you should want them:

| Trigger | When it fits |
|---|---|
| **Notebook schedule** | simplest. Fine for "score overnight". No dependency handling — if bronze is late, you score stale data and never find out. |
| **Data Pipeline** | the right default. Chain `10 → 20 → 30 → 50` as activities with success dependencies, so a failure upstream stops the run instead of quietly scoring yesterday's cohort. Add a Teams/email activity on failure. |
| **Data Activator (Reflex)** | event-driven — fires on a file landing in `Files/data/external`, rather than on a clock. Closest to "when new patient data comes in", which is what you actually want. |

For the session, a Data Pipeline is enough and it is easier to show on screen than a
scheduler.

**What must be true before you automate any of it:**

* the gates in notebook 40 pass, and `approved_by` is set — never let a schedule promote a
  model that no clinician has reviewed;
* the pipeline scores with `@champion` by alias, never a pinned version number, so promotion
  is a registry operation rather than a code change;
* every run writes an MLflow batch-inference run (§6), so scoring has the same audit trail
  as training.

**And what must not be a notebook:** single-patient, interactive scoring. A Spark session
takes tens of seconds to start — a clinician entering one patient cannot wait for that. That
is the real-time endpoint in segment 5, deployed from the same registered model. Batch runs
here; interactive runs there; both bind to `@champion`, so they cannot disagree about which
model is live.

In [ ]:
# `cohort` was set by the INCREMENTAL block above.
print(f"Pre-op patients to score: {len(cohort):,}")

X_gold = cohort[GOLD_FEATURE_COLUMNS].copy()
X_enc  = pre.transform(X_gold)

risk = model.predict_proba(X_enc)[:, 1]

# Bands agreed with the clinical team; they drive the app's triage colours.
# Thresholds are a clinical policy decision, not a modelling one — keep them visible.
BANDS = [(0.00, 0.20, "low"), (0.20, 0.40, "moderate"), (0.40, 0.60, "high"), (0.60, 1.01, "very_high")]

def band(p):
    for lo, hi, name in BANDS:
        if lo <= p < hi:
            return name
    return "unknown"

scored = pd.DataFrame({
    "episode_id":            cohort["episode_id"],
    "display_name":          cohort["display_name"],
    "provider_code":         cohort["provider_code"],
    "region":                cohort["region"],
    "age_band":              cohort["age_band"],
    "oks_t0_score":          cohort["oks_t0_score"],
    "comorbidity_count":     cohort["comorbidity_count"],
    "surgery_scheduled_date": cohort["surgery_scheduled_date"],
    "risk_poor_outcome":     risk,
    "risk_band":             [band(p) for p in risk],
    "model_name":            REGISTERED_MODEL,
    "model_version":         str(MODEL_VERSION),
    "model_alias":           CHAMPION_LABEL,
})

assert scored["risk_poor_outcome"].between(0, 1).all(), "Probabilities outside [0,1]"
print(scored["risk_band"].value_counts().to_string())
print(f"\nMean predicted risk: {risk.mean():.1%}")
scored.sort_values("risk_poor_outcome", ascending=False).head(10)

## 3. Per-patient explanations

`CalibratedClassifierCV` wraps the EBM, so the glassbox term contributions live one layer
down. We reach through to the underlying estimator and use `explain_local`, which returns
the *actual additive contributions the model used* — not an approximation of them. That is
the difference between an EBM and post-hoc SHAP, and it is worth saying on stage: for a
glassbox model the explanation is the model, so there is no "is the explanation faithful?"
question to answer.

The fallback path covers the case where the wrapped estimator is not an EBM.

In [ ]:
def underlying_ebm(m):
    """Reach through CalibratedClassifierCV to the fitted EBM, if there is one."""
    if hasattr(m, "explain_local"):
        return m
    for cc in getattr(m, "calibrated_classifiers_", []):
        est = getattr(cc, "estimator", None) or getattr(cc, "base_estimator", None)
        if hasattr(est, "explain_local"):
            return est
    return None

ebm = underlying_ebm(model)
TOP_N = 6

rows = []
if ebm is not None:
    print("Using EBM additive term contributions (exact, not post-hoc).")
    local = ebm.explain_local(X_enc)
    for i, episode_id in enumerate(cohort["episode_id"]):
        d = local.data(i)
        names, scores = d["names"], d["scores"]
        order = np.argsort(np.abs(scores))[::-1][:TOP_N]
        for rank, j in enumerate(order, start=1):
            rows.append({
                "episode_id":   episode_id,
                "rank":         rank,
                "feature":      str(names[j]),
                "feature_value": float(X_enc.iloc[i].get(names[j], np.nan))
                                 if names[j] in X_enc.columns else None,
                "contribution": float(scores[j]),
                "direction":    "increases_risk" if scores[j] > 0 else "reduces_risk",
            })
else:
    print("No EBM found — falling back to SHAP.")
    import shap
    bg = shap.sample(X_enc, 100, random_state=42)
    explainer = shap.Explainer(lambda d: model.predict_proba(d)[:, 1], bg)
    sv = explainer(X_enc)
    for i, episode_id in enumerate(cohort["episode_id"]):
        vals = sv.values[i]
        order = np.argsort(np.abs(vals))[::-1][:TOP_N]
        for rank, j in enumerate(order, start=1):
            rows.append({
                "episode_id":    episode_id,
                "rank":          rank,
                "feature":       X_enc.columns[j],
                "feature_value": float(X_enc.iloc[i, j]),
                "contribution":  float(vals[j]),
                "direction":     "increases_risk" if vals[j] > 0 else "reduces_risk",
            })

explain_df = pd.DataFrame(rows)
explain_df["model_version"] = str(MODEL_VERSION)
print(f"{len(explain_df):,} explanation rows ({TOP_N} per patient)")
explain_df.head(TOP_N)

## 4. Clinician-readable feature labels

`oks_function_subscale` means nothing at a bedside. The app renders `feature_label`, and
the Data Agent grounds on it too — an agent that answers "the t0_symptom_period term
contributed 0.08" has technically answered and practically failed.

In [ ]:
FEATURE_LABELS = {
    "oks_t0_score":          "Overall pre-op Oxford Knee Score",
    "oks_pain_subscale":     "Pre-op pain severity",
    "oks_function_subscale": "Pre-op physical function",
    "oks_adl_subscale":      "Pre-op daily activities",
    "comorbidity_count":     "Number of long-term conditions",
    "t0_symptom_period":     "How long symptoms have lasted",
    "t0_mobility":           "Mobility (EQ-5D)",
    "t0_self_care":          "Self-care (EQ-5D)",
    "t0_activity":           "Usual activities (EQ-5D)",
    "t0_discomfort":         "Pain / discomfort (EQ-5D)",
    "t0_anxiety":            "Anxiety / depression (EQ-5D)",
    "t0_disability":         "Recorded disability",
    "t0_previous_surgery":   "Previous surgery on this knee",
    "t0_assisted":           "Needed help completing the questionnaire",
    "sex_male":              "Sex: male",
    "sex_not_recorded":      "Sex not recorded (suppressed by NHS for confidentiality)",
    "depression":            "Depression",
    "diabetes":              "Diabetes",
    "heart_disease":         "Heart disease",
    "high_bp":               "High blood pressure",
    "arthritis":             "Arthritis",
    "university_hospital":   "Treated at a teaching hospital",
    "independent_hospital":  "Treated at an independent-sector provider",
}

def label_for(feature: str) -> str:
    if feature in FEATURE_LABELS:
        return FEATURE_LABELS[feature]
    # one-hot names look like "region_London" or "age_band_grouped_70 to 79"
    for prefix, human in [("region_", "Region"), ("age_band_grouped_", "Age band"),
                          ("year_", "Treatment year"), ("sex_", "Sex"),
                          ("living_arrangements_grouped_", "Living arrangements")]:
        if feature.startswith(prefix):
            return f"{human}: {feature[len(prefix):]}"
    if feature.startswith("oks_t0_"):
        return f"Pre-op OKS: {feature[len('oks_t0_'):].replace('_', ' ')}"
    return feature.replace("_", " ").capitalize()

explain_df["feature_label"] = explain_df["feature"].map(label_for)
explain_df[["feature", "feature_label", "contribution", "direction"]].head(12)

## 5. Write both tables back to gold

In [ ]:
from pyspark.sql import types as T

risk_sdf = (spark.createDataFrame(scored)
    .withColumn("_scored_at_utc", F.current_timestamp())
    .withColumn("risk_poor_outcome", F.col("risk_poor_outcome").cast(T.DoubleType())))

(risk_sdf.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
         .saveAsTable(GOLD_RISK))
spark.sql(f"OPTIMIZE {GOLD_RISK} ZORDER BY (provider_code, episode_id)")

explain_sdf = (spark.createDataFrame(explain_df)
    .withColumn("_scored_at_utc", F.current_timestamp()))

(explain_sdf.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
            .saveAsTable(GOLD_EXPLAIN))
spark.sql(f"OPTIMIZE {GOLD_EXPLAIN} ZORDER BY (episode_id)")

print(f"{GOLD_RISK}: {risk_sdf.count():,} rows")
print(f"{GOLD_EXPLAIN}: {explain_sdf.count():,} rows")

In [ ]:
def _sql_str(text: str) -> str:
    """Escape a Python string for a Spark SQL single-quoted literal.

    Spark ends the literal at the first unescaped quote -- doubling it ('') does NOT
    escape, it terminates the string and starts a new one. Backslash is the escape.
    """
    return text.replace("\\", "\\\\").replace("'", "\\'")


# Column documentation — again, this is what grounds the Data Agent.
spark.sql(f"COMMENT ON TABLE {GOLD_RISK} IS "
          "'Calibrated risk of a poor outcome after knee replacement, one row per pre-operative patient. "
          "risk_poor_outcome is a true probability: 0.34 means roughly 34 of 100 similar patients "
          "will not gain a clinically meaningful improvement. Decision support only.'")

for col, c in {
    "episode_id":            "Patient episode identifier. Joins to gold.knee_preop_cohort and gold.risk_explanation.",
    "display_name":          "Pseudonymous patient name shown in the clinician app.",
    "risk_poor_outcome":     "Calibrated probability (0-1) that this patient will NOT achieve a clinically meaningful improvement. Higher means worse expected outcome.",
    "risk_band":             "low (<20%), moderate (20-40%), high (40-60%), very_high (>60%). Clinical policy thresholds, not model output.",
    "model_version":         "Registry version of the model that produced this score.",
    "surgery_scheduled_date": "Planned date of surgery.",
    "provider_code":         "Hospital responsible for this patient. The row-level security key in the clinician app.",
}.items():
    spark.sql(f"ALTER TABLE {GOLD_RISK} CHANGE COLUMN {col} COMMENT '{_sql_str(c)}'")

spark.sql(f"COMMENT ON TABLE {GOLD_EXPLAIN} IS "
          "'Per-patient feature contributions from the glassbox EBM. Long format: six rows per patient, "
          "ranked by absolute contribution. Positive contribution increases predicted risk. "
          "Use feature_label when answering a clinician.'")

for col, c in {
    "episode_id":    "Patient episode identifier.",
    "rank":          "1 = strongest contributor for this patient.",
    "feature":       "Model feature name.",
    "feature_label": "Clinician-readable description of the feature. Prefer this in answers.",
    "contribution":  "Additive contribution to the log-odds of a poor outcome. Positive increases risk.",
    "direction":     "increases_risk or reduces_risk.",
}.items():
    spark.sql(f"ALTER TABLE {GOLD_EXPLAIN} CHANGE COLUMN {col} COMMENT '{_sql_str(c)}'")

print("Documented.")

## 6. Log the scoring run

Batch inference gets its own MLflow run. When somebody asks in six months "which model
produced the score this clinician acted on", the answer is a run id with the model version
and the cohort size on it.

In [ ]:
with mlflow.start_run(run_name="batch-scoring") as run:
    mlflow.set_tags({
        "run_type":       "batch_inference",
        "model_name":     REGISTERED_MODEL,
        "model_version":  str(MODEL_VERSION),
        "model_alias":    CHAMPION_LABEL,
        "input_table":    GOLD_COHORT,
        "output_tables":  f"{GOLD_RISK},{GOLD_EXPLAIN}",
        "risk_class":     "high_risk_eu_ai_act",
        "notebook":       "50_batch_score",
    })
    mlflow.log_metrics({
        "patients_scored":   float(len(scored)),
        "mean_risk":         float(scored["risk_poor_outcome"].mean()),
        "pct_high_or_above": float((scored["risk_band"].isin(["high", "very_high"])).mean()),
    })
    print(f"Scoring run: {run.info.run_id}")

## 7. Margaret, end to end

The rehearsal check. One query, one patient, score plus reasons — this is the shape of
what the clinician app renders and what the Data Agent should say back.

In [ ]:
MARGARET_EPISODE_ID = None   # paste the id printed by notebook 30, section 8

if MARGARET_EPISODE_ID is None:
    MARGARET_EPISODE_ID = (spark.table(GOLD_RISK)
                                .filter("display_name LIKE 'Margaret%'")
                                .orderBy(F.col("risk_poor_outcome").desc())
                                .first()["episode_id"])
    print(f"Auto-selected MARGARET_EPISODE_ID = {MARGARET_EPISODE_ID}")

display(spark.sql(f"""
    SELECT r.display_name, r.age_band, r.oks_t0_score, r.comorbidity_count,
           ROUND(r.risk_poor_outcome * 100, 1) AS risk_pct, r.risk_band,
           r.model_name, r.model_version, r.surgery_scheduled_date
    FROM {GOLD_RISK} r
    WHERE r.episode_id = '{MARGARET_EPISODE_ID}'
"""))

display(spark.sql(f"""
    SELECT rank, feature_label, ROUND(contribution, 4) AS contribution, direction
    FROM {GOLD_EXPLAIN}
    WHERE episode_id = '{MARGARET_EPISODE_ID}'
    ORDER BY rank
"""))

## 8. Cohort questions the Data Agent must get right

Rehearse these as SQL first. If the SQL is awkward, the agent will get it wrong — that is
usually a signal the gold model needs another column, not that the agent needs more
prompting.

In [ ]:
print("Q1: Which of my pre-op patients are highest risk, and why?")
display(spark.sql(f"""
    SELECT r.display_name, ROUND(r.risk_poor_outcome*100,1) AS risk_pct, r.risk_band,
           CONCAT_WS('; ', COLLECT_LIST(e.feature_label)) AS top_drivers
    FROM {GOLD_RISK} r
    JOIN {GOLD_EXPLAIN} e
      ON e.episode_id = r.episode_id AND e.rank <= 3 AND e.direction = 'increases_risk'
    GROUP BY r.display_name, r.risk_poor_outcome, r.risk_band
    ORDER BY r.risk_poor_outcome DESC
    LIMIT 10
"""))

print("Q2: How does risk vary by provider?")
display(spark.sql(f"""
    SELECT provider_code, COUNT(*) AS patients,
           ROUND(AVG(risk_poor_outcome)*100, 1) AS mean_risk_pct,
           SUM(CASE WHEN risk_band IN ('high','very_high') THEN 1 ELSE 0 END) AS high_risk_patients
    FROM {GOLD_RISK}
    GROUP BY provider_code
    HAVING COUNT(*) >= 20
    ORDER BY mean_risk_pct DESC
    LIMIT 10
"""))

print("Q3: Threshold what-if — how many patients would we flag at each cut-off?")
display(spark.sql(f"""
    SELECT t.threshold,
           SUM(CASE WHEN r.risk_poor_outcome >= t.threshold THEN 1 ELSE 0 END) AS flagged,
           ROUND(100.0 * SUM(CASE WHEN r.risk_poor_outcome >= t.threshold THEN 1 ELSE 0 END)
                 / COUNT(*), 1) AS pct_of_cohort
    FROM {GOLD_RISK} r
    CROSS JOIN (SELECT explode(array(0.2,0.3,0.4,0.5,0.6)) AS threshold) t
    GROUP BY t.threshold ORDER BY t.threshold
"""))

---

Gold now holds scores and explanations for every pre-op patient, each stamped with the
model version that produced it. The Rayfin app and the Data Agent read these tables — and
only these tables.